**(A) DATASET LINK:**

https://www.kaggle.com/datasets/abdelmalekeladjelet/sentiment-analysis-dataset

**IMPORT**

In [1]:
!pip install nltk
import nltk

# Download the lexicon
nltk.download("vader_lexicon")

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


True

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**(B.a) Vader With Sentences From the Dataset**

In [32]:
# import the lexicon
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import pandas as pd

# Create our SentimentIntensityAnalyzer
my_analyzer = SentimentIntensityAnalyzer()

# 1. Load the dataset
dataset = pd.read_csv('/content/drive/MyDrive/sentiment_data.csv')

# 2. Clean the data (Drop rows missing Comments or True Sentiment)
dataset = dataset.dropna(subset=['Comment', 'Sentiment'])
dataset['Comment'] = dataset['Comment'].astype(str)

# 3. Take a slice of the first 50 rows for testing
test_data = dataset.head(50).copy()

# 4. Helper function to translate VADER scores directly to words
def get_vader_prediction(text):
    scores = my_analyzer.polarity_scores(text)
    compound = scores['compound']

    if compound >= 0.05:
        return "Positive"
    elif compound <= -0.05:
        return "Negative"
    else:
        return "Neutral"

# 5. Apply the function to create a new 'Predicted' column
test_data['Predicted'] = test_data['Comment'].apply(get_vader_prediction)

# Rename the original sentiment column and map the numbers to actual words
label_names = {0: "Negative", 1: "Neutral", 2: "Positive"}
test_data = test_data.rename(columns={'Sentiment': 'True_Label'})
test_data['True_Label'] = test_data['True_Label'].map(label_names)

# --- DISPLAYING THE RESULTS ---

# Print the table (showing Comment, True Label, and Predicted Label as words)
print("--- FIRST 50 COMMENTS: TRUE VS PREDICTED ---\n")
print(test_data[['Comment', 'True_Label', 'Predicted']].to_string(index=False))
print("\n" + "="*50 + "\n")

# Count the occurrences
true_counts = test_data['True_Label'].value_counts()
pred_counts = test_data['Predicted'].value_counts()

print("--- LABEL COUNTS ---")
print("True Labels:")
for label, count in true_counts.items():
    print(f"  {label}: {count}")

print("\nPredicted Labels:")
for label, count in pred_counts.items():
    print(f"  {label}: {count}")

# Find the most occurrent (mode)
most_true = test_data['True_Label'].mode()[0]
most_pred = test_data['Predicted'].mode()[0]

print("\n--- MOST OCCURRENT ---")
print(f"Most common True Label: {most_true}")
print(f"Most common Predicted Label: {most_pred}\n")

# --- CONFUSION MATRIX ---
# Force the matrix to display in a logical order (Negative, Neutral, Positive)
label_order = ["Negative", "Neutral", "Positive"]
test_data['True_Label'] = pd.Categorical(test_data['True_Label'], categories=label_order, ordered=True)
test_data['Predicted'] = pd.Categorical(test_data['Predicted'], categories=label_order, ordered=True)

# Generate the Confusion Matrix using Pandas crosstab
conf_matrix = pd.crosstab(
    test_data['True_Label'],
    test_data['Predicted'],
    rownames=['ACTUAL ↓'],
    colnames=['PREDICTED →'],
    dropna=False # Ensures all columns show up even if the count is 0
)

print("--- CONFUSION MATRIX ---")
print(conf_matrix)

# Optional: Calculate Overall Accuracy from the matrix
correct_predictions = sum(conf_matrix.loc[label, label] for label in label_order if label in conf_matrix.columns and label in conf_matrix.index)
accuracy = (correct_predictions / len(test_data)) * 100
print(f"\nOverall Accuracy: {accuracy:.1f}%")

--- FIRST 50 COMMENTS: TRUE VS PREDICTED ---

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           Comment True_Label Predicted
                                                                                                                                                                                                                                                                                           lets forget apple pay required brand new iphone order use significant portion apples user base wasnt able use even wanted successive ip

Sentiment Score Guide for Vader
*   a positive sentiment has compound ≥ 0.05.
*   a negative sentiment has compound ≤ -0.05.
*   a neutral sentiment, the compound is between ]-0.05, 0.05[









In [12]:
!pip install afinn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 5.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for afinn: filename=afinn-0.1-py3-none-any.whl size=53431 sha256=0a711578157f5d57be6e53bb0f4929a3a202f01c9c0c2328328a91e3caaa6983
  Stored in directory: /root/.cache/pip/wheels/f9/72/27/74994e77200dae3d6aea2b546264500cee21f738c51241320b
Successfully built afinn


**(B.b) AFINN With Sentences From the Dataset**

In [44]:
# Import the AFINN lexicon
from afinn import Afinn
import pandas as pd

# Create our instance of AFINN
my_analyzer = Afinn()

# Update AFINN's internal dictionary with your custom values
new_values = {"pogi": 3.0, "magulo": -2.5, "pangit": -3.0, "semipogi": 1.5}
my_analyzer._dict.update(new_values)

# 1. Load the dataset
dataset = pd.read_csv('/content/drive/MyDrive/sentiment_data.csv')

# 2. Clean the data
dataset = dataset.dropna(subset=['Comment', 'Sentiment'])
dataset['Comment'] = dataset['Comment'].astype(str)

# 3. Take a slice of the first 50 rows for testing
test_data = dataset.head(50).copy()

# 4. EDIT: Helper function now returns words directly
def get_afinn_prediction(text):
    score = my_analyzer.score(text)
    if score > 0:
        return "Positive"
    elif score < 0:
        return "Negative"
    else:
        return "Neutral"

# 5. Apply the function
test_data['Predicted'] = test_data['Comment'].apply(get_afinn_prediction)

# 6. EDIT: Convert the numerical True Labels to words immediately
label_names = {0: "Negative", 1: "Neutral", 2: "Positive"}
test_data = test_data.rename(columns={'Sentiment': 'True_Label'})
test_data['True_Label'] = test_data['True_Label'].map(label_names)

# --- DISPLAYING THE RESULTS ---

print("--- FIRST 50 COMMENTS: TRUE VS PREDICTED ---\n")
# This will now show the actual words in the table
print(test_data[['Comment', 'True_Label', 'Predicted']].to_string(index=False))
print("\n" + "="*50 + "\n")

# Count the occurrences (already in word format)
true_counts = test_data['True_Label'].value_counts()
pred_counts = test_data['Predicted'].value_counts()

print("--- LABEL COUNTS ---")
print("True Labels:")
for label, count in true_counts.items():
    print(f"  {label}: {count}")

print("\nPredicted Labels:")
for label, count in pred_counts.items():
    print(f"  {label}: {count}")

# Find the most occurrent
most_true = test_data['True_Label'].mode()[0]
most_pred = test_data['Predicted'].mode()[0]

print("\n--- MOST OCCURRENT ---")
print(f"Most common True Label: {most_true}")
print(f"Most common Predicted Label: {most_pred}")

# --- CONFUSION MATRIX ---
# Force the matrix to display in a logical order
label_order = ["Negative", "Neutral", "Positive"]
test_data['True_Label'] = pd.Categorical(test_data['True_Label'], categories=label_order, ordered=True)
test_data['Predicted'] = pd.Categorical(test_data['Predicted'], categories=label_order, ordered=True)

conf_matrix = pd.crosstab(
    test_data['True_Label'],
    test_data['Predicted'],
    rownames=['ACTUAL ↓'],
    colnames=['PREDICTED →'],
    dropna=False
)

print("\n--- CONFUSION MATRIX ---")
print(conf_matrix)

# Calculate Accuracy
correct_predictions = sum(conf_matrix.loc[label, label] for label in label_order)
accuracy = (correct_predictions / len(test_data)) * 100
print(f"\nOverall Accuracy: {accuracy:.1f}%")

--- FIRST 50 COMMENTS: TRUE VS PREDICTED ---

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           Comment True_Label Predicted
                                                                                                                                                                                                                                                                                           lets forget apple pay required brand new iphone order use significant portion apples user base wasnt able use even wanted successive ip

**(B.c) TextBlob With Sentences From the Dataset**

In [49]:
# Import TextBlob
from textblob import TextBlob
import pandas as pd

# 1. Load the dataset
dataset = pd.read_csv('/content/drive/MyDrive/sentiment_data.csv')

# 2. Clean the data (Drop rows missing Comments or True Sentiment)
dataset = dataset.dropna(subset=['Comment', 'Sentiment'])
dataset['Comment'] = dataset['Comment'].astype(str)

# 3. Take a slice of the first 50 rows for testing
test_data = dataset.head(50).copy()

# 4. Helper function to translate TextBlob scores directly to words
def get_textblob_prediction(text):
    # TextBlob calculates sentiment polarity between -1.0 (Negative) and 1.0 (Positive)
    score = TextBlob(text).sentiment.polarity

    if score > 0:
        return "Positive"
    elif score < 0:
        return "Negative"
    else:
        return "Neutral"

# 5. Apply the function to create a new 'Predicted' column
test_data['Predicted'] = test_data['Comment'].apply(get_textblob_prediction)

# Rename the original sentiment column and map the numbers to actual words
label_names = {0: "Negative", 1: "Neutral", 2: "Positive"}
test_data = test_data.rename(columns={'Sentiment': 'True_Label'})
test_data['True_Label'] = test_data['True_Label'].map(label_names)

# --- DISPLAYING THE RESULTS ---

# Print the table (showing Comment, True Label, and Predicted Label as words)
print("--- FIRST 50 COMMENTS: TRUE VS PREDICTED ---\n")
print(test_data[['Comment', 'True_Label', 'Predicted']].to_string(index=False))
print("\n" + "="*50 + "\n")

# Count the occurrences
true_counts = test_data['True_Label'].value_counts()
pred_counts = test_data['Predicted'].value_counts()

print("--- LABEL COUNTS ---")
print("True Labels:")
for label, count in true_counts.items():
    print(f"  {label}: {count}")

print("\nPredicted Labels:")
for label, count in pred_counts.items():
    print(f"  {label}: {count}")

# Find the most occurrent (mode)
most_true = test_data['True_Label'].mode()[0]
most_pred = test_data['Predicted'].mode()[0]

print("\n--- MOST OCCURRENT ---")
print(f"Most common True Label: {most_true}")
print(f"Most common Predicted Label: {most_pred}\n")

# --- CONFUSION MATRIX ---
# Force the matrix to display in a logical order (Negative, Neutral, Positive)
label_order = ["Negative", "Neutral", "Positive"]
test_data['True_Label'] = pd.Categorical(test_data['True_Label'], categories=label_order, ordered=True)
test_data['Predicted'] = pd.Categorical(test_data['Predicted'], categories=label_order, ordered=True)

# Generate the Confusion Matrix using Pandas crosstab
conf_matrix = pd.crosstab(
    test_data['True_Label'],
    test_data['Predicted'],
    rownames=['ACTUAL ↓'],
    colnames=['PREDICTED →'],
    dropna=False # Ensures all columns show up even if the count is 0
)

print("--- CONFUSION MATRIX ---")
print(conf_matrix)

# Calculate Overall Accuracy from the matrix
correct_predictions = sum(conf_matrix.loc[label, label] for label in label_order if label in conf_matrix.columns and label in conf_matrix.index)
accuracy = (correct_predictions / len(test_data)) * 100
print(f"\nOverall Accuracy: {accuracy:.1f}%")

--- FIRST 50 COMMENTS: TRUE VS PREDICTED ---

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           Comment True_Label Predicted
                                                                                                                                                                                                                                                                                           lets forget apple pay required brand new iphone order use significant portion apples user base wasnt able use even wanted successive ip

**VADER Sentiment Analysis**

In [51]:
import nltk
import pandas as pd
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.tokenize import word_tokenize

# Download the required tokenizer package for NLTK
nltk.download('punkt', quiet=True)

# Create our instance of SentimentIntensityAnalyzer
my_analyzer = SentimentIntensityAnalyzer()

# Update lexicon with your custom values
new_values = {"pogi": 3.0, "magulo": -2.5, "pangit": -3.0, "semipogi": 1.5}
my_analyzer.lexicon.update(new_values)

# 1. Load and clean the dataset
dataset = pd.read_csv('/content/drive/MyDrive/sentiment_data.csv')
dataset = dataset.dropna(subset=['Comment', 'Sentiment'])
dataset['Comment'] = dataset['Comment'].astype(str)

# 2. Take a small slice of the first 5 rows for testing
test_data = dataset.head(5).copy()

print("=== WORD VALENCE BREAKDOWN FOR FIRST 5 COMMENTS ===\n")

# 3. Loop through each comment in our sample
for index, row in test_data.iterrows():
    comment = row['Comment']
    true_label = row['Sentiment']

    # Tokenize the comment
    tokenized_sentence = word_tokenize(comment)

    pos_word_list = []
    neu_word_list = []
    neg_word_list = []

    for word in tokenized_sentence:
        # Look up the exact valence score in VADER's dictionary
        valence_score = my_analyzer.lexicon.get(word.lower(), 0.0)

        # Sort based on the raw valence score
        if valence_score > 0:
            pos_word_list.append((word, valence_score))
        elif valence_score < 0:
            neg_word_list.append((word, valence_score))
        else:
            neu_word_list.append((word, valence_score))

    # Print the breakdown for this specific comment
    print(f"COMMENT {index + 1}: '{comment}'")
    print(f"True Label in Dataset: {true_label}")
    print(f"Positive Words: {pos_word_list}")
    print(f"Neutral Words:  {neu_word_list}")
    print(f"Negative Words: {neg_word_list}")

    # Print the overall VADER compound score
    overall_score = my_analyzer.polarity_scores(comment)
    print(f"Overall VADER Score: {overall_score['compound']}\n")
    print("-" * 60 + "\n")

=== WORD VALENCE BREAKDOWN FOR FIRST 5 COMMENTS ===

COMMENT 1: 'lets forget apple pay required brand new iphone order use significant portion apples user base wasnt able use even wanted successive iphone incorporated technology older iphones replaced number people could use technology increased'
True Label in Dataset: 1
Positive Words: [('significant', 0.8), ('successive', 1.1), ('number', 0.3), ('increased', 1.1)]
Neutral Words:  [('lets', 0.0), ('apple', 0.0), ('required', 0.0), ('brand', 0.0), ('new', 0.0), ('iphone', 0.0), ('order', 0.0), ('use', 0.0), ('portion', 0.0), ('apples', 0.0), ('user', 0.0), ('base', 0.0), ('wasnt', 0.0), ('able', 0.0), ('use', 0.0), ('even', 0.0), ('wanted', 0.0), ('iphone', 0.0), ('incorporated', 0.0), ('technology', 0.0), ('older', 0.0), ('iphones', 0.0), ('replaced', 0.0), ('people', 0.0), ('could', 0.0), ('use', 0.0), ('technology', 0.0)]
Negative Words: [('forget', -0.9), ('pay', -0.4)]
Overall VADER Score: 0.4588

---------------------------------

**AFINN Sentiment Analysis**

In [50]:
# Import the AFINN lexicon
from afinn import Afinn
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize

# Download the required tokenizer package for NLTK
nltk.download('punkt', quiet=True)

# Create our instance of AFINN
my_analyzer = Afinn()

# Update AFINN's internal dictionary with your custom values
new_values = {"pogi": 3.0, "magulo": -2.5, "pangit": -3.0, "semipogi": 1.5}
my_analyzer._dict.update(new_values)

# 1. Load and clean the dataset
dataset = pd.read_csv('/content/drive/MyDrive/sentiment_data.csv')
dataset = dataset.dropna(subset=['Comment', 'Sentiment'])
dataset['Comment'] = dataset['Comment'].astype(str)

# 2. Take a small slice of the first 5 rows for testing
test_data = dataset.head(5).copy()

# Helper for labeling
label_names = {0: "Negative", 1: "Neutral", 2: "Positive"}

print("=== WORD VALENCE BREAKDOWN (AFINN) FOR FIRST 5 COMMENTS ===\n")

# 3. Loop through each comment in our sample
for index, row in test_data.iterrows():
    comment = row['Comment']
    true_label_num = row['Sentiment']
    true_label_name = label_names.get(true_label_num, "Unknown")

    # Tokenize the comment
    tokenized_sentence = word_tokenize(comment)

    pos_word_list = []
    neu_word_list = []
    neg_word_list = []

    for word in tokenized_sentence:
        # Look up the exact valence score in AFINN's dictionary
        # We use word.lower() because AFINN dictionary keys are lowercase
        valence_score = my_analyzer._dict.get(word.lower(), 0.0)

        # Sort based on the raw valence score
        if valence_score > 0:
            pos_word_list.append((word, valence_score))
        elif valence_score < 0:
            neg_word_list.append((word, valence_score))
        else:
            neu_word_list.append((word, valence_score))

    # Print the breakdown for this specific comment
    print(f"COMMENT {index + 1}: '{comment}'")
    print(f"True Label in Dataset: {true_label_name} ({true_label_num})")
    print(f"Positive Words: {pos_word_list}")
    print(f"Neutral Words:  {neu_word_list}")
    print(f"Negative Words: {neg_word_list}")

    # Print the overall AFINN score (sum of all word scores)
    overall_score = my_analyzer.score(comment)
    print(f"Overall AFINN Score: {overall_score}\n")
    print("-" * 60 + "\n")

=== WORD VALENCE BREAKDOWN (AFINN) FOR FIRST 5 COMMENTS ===

COMMENT 1: 'lets forget apple pay required brand new iphone order use significant portion apples user base wasnt able use even wanted successive iphone incorporated technology older iphones replaced number people could use technology increased'
True Label in Dataset: Neutral (1)
Positive Words: [('significant', 1), ('increased', 1)]
Neutral Words:  [('lets', 0.0), ('apple', 0.0), ('required', 0.0), ('brand', 0.0), ('new', 0.0), ('iphone', 0.0), ('order', 0.0), ('use', 0.0), ('portion', 0.0), ('apples', 0.0), ('user', 0.0), ('base', 0.0), ('wasnt', 0.0), ('able', 0.0), ('use', 0.0), ('even', 0.0), ('wanted', 0.0), ('successive', 0.0), ('iphone', 0.0), ('incorporated', 0.0), ('technology', 0.0), ('older', 0.0), ('iphones', 0.0), ('replaced', 0.0), ('number', 0.0), ('people', 0.0), ('could', 0.0), ('use', 0.0), ('technology', 0.0)]
Negative Words: [('forget', -1), ('pay', -1)]
Overall AFINN Score: 0.0

--------------------------

**TEXTBLOB Sentiment Analysis**

In [46]:
# Import TextBlob
from textblob import TextBlob
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize

# Download the required tokenizer package for NLTK
nltk.download('punkt', quiet=True)

# 1. Load and clean the dataset
dataset = pd.read_csv('/content/drive/MyDrive/sentiment_data.csv')
dataset = dataset.dropna(subset=['Comment', 'Sentiment'])
dataset['Comment'] = dataset['Comment'].astype(str)

# 2. Take a small slice of the first 5 rows for testing
test_data = dataset.head(5).copy()

# Helper for labeling
label_names = {0: "Negative", 1: "Neutral", 2: "Positive"}

print("=== WORD VALENCE BREAKDOWN (TEXTBLOB) FOR FIRST 5 COMMENTS ===\n")

# 3. Loop through each comment in our sample
for index, row in test_data.iterrows():
    comment = row['Comment']
    true_label_num = row['Sentiment']
    true_label_name = label_names.get(true_label_num, "Unknown")

    # Tokenize the comment
    tokenized_sentence = word_tokenize(comment)

    pos_word_list = []
    neu_word_list = []
    neg_word_list = []

    for word in tokenized_sentence:
        # Get the polarity score for the individual word using TextBlob
        # TextBlob polarity ranges from -1.0 to 1.0
        word_polarity = TextBlob(word).sentiment.polarity

        # Sort based on the polarity score
        if word_polarity > 0:
            pos_word_list.append((word, word_polarity))
        elif word_polarity < 0:
            neg_word_list.append((word, word_polarity))
        else:
            neu_word_list.append((word, word_polarity))

    # Print the breakdown for this specific comment
    print(f"COMMENT {index + 1}: '{comment}'")
    print(f"True Label in Dataset: {true_label_name} ({true_label_num})")
    print(f"Positive Words: {pos_word_list}")
    print(f"Neutral Words:  {neu_word_list}")
    print(f"Negative Words: {neg_word_list}")

    # Print the overall TextBlob polarity score for the full comment
    overall_polarity = TextBlob(comment).sentiment.polarity
    print(f"Overall TextBlob Polarity: {overall_polarity}\n")
    print("-" * 60 + "\n")

=== WORD VALENCE BREAKDOWN (TEXTBLOB) FOR FIRST 5 COMMENTS ===

COMMENT 1: 'lets forget apple pay required brand new iphone order use significant portion apples user base wasnt able use even wanted successive iphone incorporated technology older iphones replaced number people could use technology increased'
True Label in Dataset: Neutral (1)
Positive Words: [('new', 0.13636363636363635), ('significant', 0.375), ('able', 0.5), ('older', 0.16666666666666666)]
Neutral Words:  [('lets', 0.0), ('forget', 0.0), ('apple', 0.0), ('pay', 0.0), ('required', 0.0), ('brand', 0.0), ('iphone', 0.0), ('order', 0.0), ('use', 0.0), ('portion', 0.0), ('apples', 0.0), ('user', 0.0), ('wasnt', 0.0), ('use', 0.0), ('even', 0.0), ('wanted', 0.0), ('successive', 0.0), ('iphone', 0.0), ('incorporated', 0.0), ('technology', 0.0), ('iphones', 0.0), ('replaced', 0.0), ('number', 0.0), ('people', 0.0), ('could', 0.0), ('use', 0.0), ('technology', 0.0), ('increased', 0.0)]
Negative Words: [('base', -0.8)]
Overall 

**(C & D) CONCLUSION**

In [48]:
import pandas as pd
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from afinn import Afinn
from textblob import TextBlob
from nltk.tokenize import word_tokenize

# Download the required tokenizer package for NLTK
nltk.download('punkt', quiet=True)

# 1. Initialize all analyzers
vader_analyzer = SentimentIntensityAnalyzer()
afinn_analyzer = Afinn()
# Adding custom values to analyzers that support it
custom_values = {"pogi": 3.0, "magulo": -2.5, "pangit": -3.0, "semipogi": 1.5}
vader_analyzer.lexicon.update(custom_values)
afinn_analyzer._dict.update(custom_values)

# 2. Load and clean the dataset
dataset = pd.read_csv('/content/drive/MyDrive/sentiment_data.csv')
dataset = dataset.dropna(subset=['Comment', 'Sentiment'])
dataset['Comment'] = dataset['Comment'].astype(str)

# 3. Take a slice for testing
test_data = dataset.head(50).copy()

# Helper for labeling
label_names = {0: "Negative", 1: "Neutral", 2: "Positive"}
test_data['True_Label_Name'] = test_data['Sentiment'].map(label_names)

# --- PREDICTION FUNCTIONS ---

def get_vader_prediction(text):
    scores = vader_analyzer.polarity_scores(text)
    compound = scores['compound']
    if compound >= 0.05: return "Positive"
    elif compound <= -0.05: return "Negative"
    else: return "Neutral"

def get_afinn_prediction(text):
    score = afinn_analyzer.score(text)
    if score > 0: return "Positive"
    elif score < 0: return "Negative"
    else: return "Neutral"

def get_textblob_prediction(text):
    score = TextBlob(text).sentiment.polarity
    if score > 0: return "Positive"
    elif score < 0: return "Negative"
    else: return "Neutral"

# 4. Generate predictions for all three
test_data['VADER_Pred'] = test_data['Comment'].apply(get_vader_prediction)
test_data['AFINN_Pred'] = test_data['Comment'].apply(get_afinn_prediction)
test_data['TextBlob_Pred'] = test_data['Comment'].apply(get_textblob_prediction)

# --- COMPARISON SUMMARY ---

def calculate_accuracy(true_labels, pred_labels):
    correct = (true_labels == pred_labels).sum()
    return (correct / len(true_labels)) * 100

vader_acc = calculate_accuracy(test_data['True_Label_Name'], test_data['VADER_Pred'])
afinn_acc = calculate_accuracy(test_data['True_Label_Name'], test_data['AFINN_Pred'])
textblob_acc = calculate_accuracy(test_data['True_Label_Name'], test_data['TextBlob_Pred'])

print("=== FINAL COMPARISON SUMMARY (First 50 Comments) ===\n")
print(f"{'Comment (First 30 chars)':<35} | {'True':<10} | {'VADER':<10} | {'AFINN':<10} | {'TextBlob':<10}")
print("-" * 95)

# Iterating through all 50 rows in test_data
for _, row in test_data.iterrows():
    comment_trunc = (row['Comment'][:32] + '..') if len(row['Comment']) > 32 else row['Comment']
    print(f"{comment_trunc:<35} | {row['True_Label_Name']:<10} | {row['VADER_Pred']:<10} | {row['AFINN_Pred']:<10} | {row['TextBlob_Pred']:<10}")

print("\n" + "="*50)
print("ACCURACY COMPARISON")
print("="*50)
print(f"VADER Accuracy:    {vader_acc:.1f}%")
print(f"AFINN Accuracy:    {afinn_acc:.1f}%")
print(f"TextBlob Accuracy: {textblob_acc:.1f}%")

# Determine winner
results = {"VADER": vader_acc, "AFINN": afinn_acc, "TextBlob": textblob_acc}
winner = max(results, key=results.get)
print(f"\nCONCLUSION: The most accurate lexicon for this sample is {winner}.")

=== FINAL COMPARISON SUMMARY (First 50 Comments) ===

Comment (First 30 chars)            | True       | VADER      | AFINN      | TextBlob  
-----------------------------------------------------------------------------------------------
lets forget apple pay required b..  | Neutral    | Positive   | Neutral    | Positive  
nz retailers don’t even contactl..  | Negative   | Positive   | Positive   | Positive  
forever acknowledge channel help..  | Positive   | Positive   | Positive   | Neutral   
whenever go place doesn’t take a..  | Negative   | Negative   | Negative   | Negative  
apple pay convenient secure easy..  | Positive   | Positive   | Positive   | Positive  
we’ve hounding bank adopt apple ..  | Neutral    | Positive   | Positive   | Positive  
got apple pay south africa it’s ..  | Positive   | Positive   | Neutral    | Negative  
need apple pay physical credit c..  | Neutral    | Positive   | Negative   | Neutral   
united states abundance retailer..  | Positive   | Positiv